Combine all five products at user level
This starts with all users, then attaches their product activity. That matters because a user who never traded in a particular product should still remain in the analysis with zero revenue for that product.

In [0]:
%sql
CREATE OR REPLACE TABLE groww_user_value AS

SELECT
    u.third_party_id,

    -- Revenue from each product
    COALESCE(s.total_brokerage, 0) AS stocks_revenue,
    COALESCE(f.total_brokerage, 0) AS fno_revenue,
    COALESCE(m.total_brokerage, 0) AS mf_revenue,
    COALESCE(i.total_brokerage, 0) AS ipo_revenue,
    COALESCE(c.total_brokerage, 0) AS commodity_revenue,

    -- Total revenue generated by each user
    COALESCE(s.total_brokerage, 0)
      + COALESCE(f.total_brokerage, 0)
      + COALESCE(m.total_brokerage, 0)
      + COALESCE(i.total_brokerage, 0)
      + COALESCE(c.total_brokerage, 0) AS total_revenue,

    -- Product activity
    COALESCE(s.total_successful_orders, 0) AS stocks_orders,
    COALESCE(f.total_executed_orders, 0) AS fno_orders,
    COALESCE(m.total_completed_orders, 0) AS mf_orders,
    COALESCE(i.total_applications, 0) AS ipo_applications,
    COALESCE(c.total_executed_orders, 0) AS commodity_orders,

    -- Active days by product
    COALESCE(s.active_trading_days, 0) AS stocks_active_days,
    COALESCE(f.active_trading_days, 0) AS fno_active_days,
    COALESCE(m.active_investment_days, 0) AS mf_active_days,
    COALESCE(i.application_days, 0) AS ipo_active_days,
    COALESCE(c.active_trading_days, 0) AS commodity_active_days

FROM (SELECT DISTINCT third_party_id FROM users
      WHERE third_party_id IS NOT NULL) u

LEFT JOIN stocks_user_metrics s
    ON u.third_party_id = s.third_party_id

LEFT JOIN fno_user_metrics f
    ON u.third_party_id = f.third_party_id

LEFT JOIN mf_user_metrics m
    ON u.third_party_id = m.third_party_id

LEFT JOIN ipo_user_metrics i
    ON u.third_party_id = i.third_party_id

LEFT JOIN commodity_user_metrics c
    ON u.third_party_id = c.third_party_id;

In [0]:
%sql
SELECT
    COUNT(*) AS total_users,
    COUNT_IF(total_revenue > 0) AS transacting_users,
    COUNT_IF(total_revenue = 0) AS non_transacting_users,
    SUM(total_revenue) AS total_groww_revenue,
    AVG(total_revenue) AS avg_revenue_per_user,
    PERCENTILE_APPROX(total_revenue, 0.50) AS median_revenue,
    PERCENTILE_APPROX(total_revenue, 0.80) AS p80_revenue,
    PERCENTILE_APPROX(total_revenue, 0.90) AS p90_revenue
FROM groww_user_value;

find which products contribute most to total revenue:

In [0]:
%sql
SELECT
    'Stocks' AS product,
    SUM(stocks_revenue) AS revenue
FROM groww_user_value

UNION ALL

SELECT 'F&O', SUM(fno_revenue)
FROM groww_user_value

UNION ALL

SELECT 'Mutual Funds', SUM(mf_revenue)
FROM groww_user_value

UNION ALL

SELECT 'IPO', SUM(ipo_revenue)
FROM groww_user_value

UNION ALL

SELECT 'Commodities', SUM(commodity_revenue)
FROM groww_user_value;

check how many products each user uses and their revenue:

In [0]:
%sql
SELECT
    CASE
        WHEN product_count = 1 THEN '1 product'
        WHEN product_count = 2 THEN '2 products'
        WHEN product_count = 3 THEN '3 products'
        WHEN product_count = 4 THEN '4 products'
        ELSE '5 products'
    END AS product_adoption,
    COUNT(*) AS users,
    ROUND(AVG(total_revenue), 2) AS avg_revenue_per_user,
    ROUND(SUM(total_revenue), 2) AS total_revenue
FROM (
    SELECT
        *,
        (CASE WHEN stocks_orders > 0 THEN 1 ELSE 0 END
        + CASE WHEN fno_orders > 0 THEN 1 ELSE 0 END
        + CASE WHEN mf_orders > 0 THEN 1 ELSE 0 END
        + CASE WHEN ipo_applications > 0 THEN 1 ELSE 0 END
        + CASE WHEN commodity_orders > 0 THEN 1 ELSE 0 END
        ) AS product_count
    FROM groww_user_value
    WHERE total_revenue > 0
)
GROUP BY product_count
ORDER BY product_count;

Identify high-value users across Groww

In [0]:
%sql
CREATE OR REPLACE TABLE groww_user_segments AS
SELECT
    *,
    NTILE(5) OVER (
        ORDER BY total_revenue DESC, third_party_id
    ) AS value_segment
FROM groww_user_value
WHERE total_revenue > 0;

In [0]:
%sql
SELECT
    value_segment,
    COUNT(*) AS users,
    ROUND(AVG(total_revenue), 2) AS avg_revenue_per_user,
    ROUND(SUM(total_revenue), 2) AS total_revenue,
    ROUND(
        100.0 * SUM(total_revenue)
        / SUM(SUM(total_revenue)) OVER (),
        2
    ) AS revenue_contribution_pct
FROM groww_user_segments
GROUP BY value_segment
ORDER BY value_segment;

High-value users are users who generate disproportionately high revenue for Groww across its product ecosystem. We will use total revenue as the primary measure and product adoption, activity, and transaction behaviour as supporting characteristics.

The final answer should establish:
1. Revenue concentration: What percentage of total revenue comes from the top 20% of transacting users?
2. Behavioural differences: Do high-value users transact more frequently, use more products, or remain active across more days?
3. Product mix: Which products contribute most to revenue, and which products are commonly used by high-value users?
One analytical caveat: F&O, Commodities, MF and IPO revenue are fixed fees per successful transaction under our assignment rules. Therefore, higher transaction counts mechanically produce higher revenue in those products. We should not present order count alone as independent proof of customer value

-----------------------------------------------------------------------
What criteria did you use to identify high-value users?
I used the following factors:
Factor	Why I considered it
Total revenue	Primary measure of direct business value
Transaction frequency	Indicates how frequently users generate revenue
Product adoption	Indicates engagement across Groww's product ecosystem
Active trading/investment days	Indicates consistency of engagement
Product-wise revenue contribution	Identifies which products drive business value

I used total revenue as the primary segmentation criterion, while the other factors helped explain user behaviour.

I ranked all users with positive revenue and divided them into five equal-sized groups using NTILE(5).

The top 20% were classified as high-value users.

---------------------------------------------------------------------
What evidence supported this decision?
Our results showed:
- Top 20% of users generated 97.39% of total revenue.
- Their average revenue per user was ₹2,239.28.
- The remaining 80% generated only 2.61% of revenue.
- F&O contributed approximately 76% of total revenue.
- Multi-product users generated substantially higher average revenue than single-product users.